In [ ]:
import pandas as pd

df_vnx = pd.read_excel(r'D:\Downloads\DSTC vòng 3\cleaned data\UPCOM_cleaned.xlsx')
df_vnx.drop("Unnamed: 0", axis =1, inplace = True)
df_vnx.info()

In [ ]:
df_vnx['min_price_in_future'] = df_vnx.groupby('ticker')['low'].shift(-10).rolling(window=10).min()
df_vnx['future_max_drawdown'] = (df_vnx['min_price_in_future'] / df_vnx['close']) - 1
df_vnx.dropna(subset=['future_max_drawdown'], inplace=True)


df_vnx['T2_drawdown'] = df_vnx.groupby('timestamp')['future_max_drawdown'].transform(lambda x: x.quantile(0.25))


In [ ]:
print(df_vnx['future_max_drawdown'].describe(percentiles=[0.1, 0.25,0.3, 0.5, 0.7, 0.75, 0.9]))

In [ ]:
import numpy as np

# conditions = [
#     df_vnx['future_max_drawdown'] <= df_vnx['T1_drawdown'], 
#     (df_vnx['future_max_drawdown'] > df_vnx['T1_drawdown']) & (df_vnx['future_max_drawdown'] <= df_vnx['T2_drawdown']), 
#     df_vnx['future_max_drawdown'] > df_vnx['T2_drawdown'] 
# ]

# labels = [2,1,0] 


conditions = [
    df_vnx['future_max_drawdown'] <= df_vnx['T2_drawdown'],  
    df_vnx['future_max_drawdown'] > df_vnx['T2_drawdown'] 
]


labels = [1,0] 
df_vnx['risk_label'] = np.select(conditions, labels)

df_vnx.head()

In [ ]:
df_vnx.dropna(axis = 0, how = 'any', inplace = True)
df_vnx.tail()

In [ ]:
df_vnx["risk_label"].value_counts()

In [ ]:
df = df_vnx
df.reset_index(inplace = True)
df.drop("index", axis =1, inplace = True)
df.info()

In [ ]:
import numpy as np
import pandas as pd

def add_features(df):
    df = df.copy()

    df["volatility_5d"] = df.groupby("ticker")["return"].rolling(5).std().reset_index(0, drop=True)
    df["volatility_10d"] = df.groupby("ticker")["return"].rolling(10).std().reset_index(0, drop=True)

    df["volume_change"] = df.groupby("ticker")["volume"].pct_change()
    
    df["obv"] = (np.sign(df["return"].fillna(0)) * df["volume"]).groupby(df["ticker"]).cumsum()
    
    df["volume_pct_20d"] = df.groupby("ticker")["volume"].transform(
        lambda x: x.rolling(20).apply(lambda s: pd.Series(s).rank(pct=True).iloc[-1])
    )

    lag_cols = ["close", "return", "rsi", "macd", "macd_diff", "bollinger_pct"]
    for col in lag_cols:
        for lag in [1, 2, 3]:
            df[f"{col}_lag{lag}"] = df.groupby("ticker")[col].shift(lag)
    
    return df

df = add_features(df)


df.dropna(inplace=True)

print("Số feature sau khi thêm:", df.shape[1])
df.head()


In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import ExtraTreesClassifier
import torch
import torch.nn as nn
import pickle
from tqdm import tqdm
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings('ignore')

Features = [
       'open', 'high', 'low', 'close', 'volume',
       'ema_50', 'ema_200', 'macd', 'macd_signal', 'macd_diff', 'rsi',
       'bollinger_hband', 'bollinger_lband', 'mfi',
       'return', 'bollinger_pct', 'bollinger_bw',
       'volatility_5d', 'volatility_10d',
       'volume_change', 'obv', 'volume_pct_20d', 'close_lag1', 'close_lag2',
       'close_lag3', 'return_lag1', 'return_lag2', 'return_lag3', 'rsi_lag1',
       'rsi_lag2', 'rsi_lag3', 'macd_lag1', 'macd_lag2', 'macd_lag3',
       'macd_diff_lag1', 'macd_diff_lag2', 'macd_diff_lag3',
       'bollinger_pct_lag1', 'bollinger_pct_lag2', 'bollinger_pct_lag3'
]


# Features = [
#        'open', 'high', 'low', 'close', 'volume',
#        'ema_50', 'ema_200', 'macd', 'macd_signal', 'rsi',
#        'bollinger_hband', 'bollinger_lband', 'mfi', 'return', 'bollinger_pct', 'bollinger_bw'
#        ]

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, f1_score
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier
import lightgbm as lgb

test_size = 0.2
random_state = 42
all_Xtrain_dfs = []
all_ytrain_dfs = []
all_Xtest_dfs = []
all_ytest_dfs = []

for ticker in df["ticker"].unique():
    df_ticker = df[df["ticker"] == ticker].copy()
    df_ticker.sort_values(by = "timestamp", inplace = True)
    split_index = int((len(df_ticker) * (1-test_size)))
    df_ticker_train = df_ticker[:split_index]
    df_ticker_test = df_ticker[split_index:]
    all_Xtrain_dfs.append(df_ticker_train[Features])
    all_Xtest_dfs.append(df_ticker_test[Features])
    all_ytrain_dfs.append(df_ticker_train["risk_label"])
    all_ytest_dfs.append(df_ticker_test["risk_label"])


X_train = pd.concat(all_Xtrain_dfs)
X_test = pd.concat(all_Xtest_dfs)
y_train = pd.concat(all_ytrain_dfs)
y_test = pd.concat(all_ytest_dfs)

print(f"Kích thước tập Train: X={X_train.shape}, y={y_train.shape}")
print(f"Kích thước tập Test: X={X_test.shape}, y={y_test.shape}")



In [ ]:
!pip install imblearn

In [ ]:

from imblearn.combine import SMOTEENN

smote_enn = SMOTEENN(random_state=42)
X_train_smote, y_train_smote = smote_enn.fit_resample(X_train, y_train)


print(f"\nKích thước tập Train SAU KHI SMOTE: X={X_train_smote.shape}, y={y_train_smote.shape}")
print("\nPhân bổ lớp trên tập Train SAU KHI SMOTE:")
print(y_train_smote.value_counts(normalize=True))

In [ ]:
import lightgbm as lgb
from collections import Counter
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

lgbm_classifier = lgb.LGBMClassifier(random_state=42, is_unbalance=True)
# counter = Counter(y_train)
# neg_count = counter[0]
# pos_count = counter[1]


# scale_pos_weight_value = neg_count / pos_count

# print(f"Số mẫu lớp 0 (Không rủi ro cao): {neg_count}")
# print(f"Số mẫu lớp 1 (Rủi ro cao): {pos_count}")
# print(f"Giá trị scale_pos_weight được tính toán: {scale_pos_weight_value:.4f}")



lgbm_classifier_scaled = lgb.LGBMClassifier(random_state=42)


lgbm_classifier_scaled.fit(X_train, y_train)

model_filename = 'lgbm_model_upcom.pkl'
with open(model_filename, 'wb') as file:
    pickle.dump(lgbm_classifier_scaled, file)

y_pred_proba = lgbm_classifier_scaled.predict_proba(X_test)[:, 1]
y_pred_scaled = (y_pred_proba > 0.4).astype(int)


print(classification_report(y_test, y_pred_scaled, digits=4))
cm = confusion_matrix(y_test, y_pred_scaled)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Dự đoán (Predicted)')
plt.ylabel('Thực tế (Actual)')
plt.title('Ma trận Nhầm lẫn trên tập Test')
plt.show()